In [4]:
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text

PROJECT_ID = "olist-data-pipeline-507001"
DATASET = "olist_mart"


engine = create_engine(
    f"bigquery://{PROJECT_ID}"
)

# -----------------------------
# Category GMV Spike Analysis
# -----------------------------

def get_category_spike(
    engine,
    project_id,
    dataset,
    year,
    base_month,
    comparison_month,
):

    category_spike_sql = text(f"""
        WITH category_monthly AS (

            SELECT
                d.month,

                COALESCE(
                    p.product_category_name_english,
                    'Unknown'
                ) AS product_category,

                COUNT(*) AS units_sold,

                COUNT(
                    DISTINCT i.order_key
                ) AS order_count,

                SUM(
                    i.price + i.freight_value
                ) AS gmv

            FROM `{project_id}.{dataset}.fact_order_items` AS i

            JOIN `{project_id}.{dataset}.dim_date` AS d
                ON i.order_date_key = d.date_key

            LEFT JOIN `{project_id}.{dataset}.dim_product` AS p
                ON i.product_key = p.product_key

            WHERE d.year = :year
              AND d.month IN (:base_month, :comparison_month)

            GROUP BY
                d.month,
                product_category
        ),

        comparison AS (

            SELECT
                product_category,

                SUM(
                    CASE
                        WHEN month = :base_month
                        THEN gmv
                        ELSE 0
                    END
                ) AS base_gmv,

                SUM(
                    CASE
                        WHEN month = :comparison_month
                        THEN gmv
                        ELSE 0
                    END
                ) AS comparison_gmv,

                SUM(
                    CASE
                        WHEN month = :base_month
                        THEN order_count
                        ELSE 0
                    END
                ) AS base_orders,

                SUM(
                    CASE
                        WHEN month = :comparison_month
                        THEN order_count
                        ELSE 0
                    END
                ) AS comparison_orders

            FROM category_monthly

            GROUP BY
                product_category
        ),

        uplift AS (

            SELECT
                *,

                comparison_gmv - base_gmv AS gmv_increase,

                SAFE_MULTIPLY(
                    SAFE_DIVIDE(
                        comparison_gmv - base_gmv,
                        base_gmv
                    ),
                    100
                ) AS growth_pct

            FROM comparison
        )

        SELECT
            product_category,

            ROUND(base_gmv, 2) AS base_gmv,

            ROUND(comparison_gmv, 2) AS comparison_gmv,

            ROUND(
                gmv_increase,
                2
            ) AS gmv_increase,

            ROUND(
                growth_pct,
                2
            ) AS growth_pct,

            base_orders,

            comparison_orders,

            ROUND(
                100 * SAFE_DIVIDE(
                    gmv_increase,
                    SUM(gmv_increase) OVER ()
                ),
                2
            ) AS contribution_to_spike_pct

        FROM uplift

        WHERE gmv_increase > 0

        ORDER BY
            gmv_increase DESC
    """)

    category_spike = pd.read_sql(
        category_spike_sql,
        con=engine,
        params={
            "year": year,
            "base_month": base_month,
            "comparison_month": comparison_month,
        },
    )

    return category_spike


# -----------------------------
# Streamlit interface
# -----------------------------

st.title("Category GMV Spike Analysis")

st.write(
    "Identify which product categories contributed most "
    "to the increase in GMV between two selected months."
)


# Month names for the dropdowns
month_names = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December",
}


# -----------------------------
# User selections
# -----------------------------

year = st.selectbox(
    "Select year",
    [2017, 2018],
)


base_month = st.selectbox(
    "Compare from",
    options=list(month_names.keys()),
    format_func=lambda x: month_names[x],
    index=9,
)


comparison_month = st.selectbox(
    "Compare to",
    options=list(month_names.keys()),
    format_func=lambda x: month_names[x],
    index=10,
)


# -----------------------------
# Run analysis
# -----------------------------

if base_month == comparison_month:

    st.warning(
        "Please select two different months."
    )

else:

    category_spike = get_category_spike(
        engine=engine,
        project_id=PROJECT_ID,
        dataset=DATASET,
        year=year,
        base_month=base_month,
        comparison_month=comparison_month,
    )


    # -----------------------------
    # Results
    # -----------------------------

    base_name = month_names[base_month]
    comparison_name = month_names[comparison_month]


    st.subheader(
        f"{base_name} → {comparison_name} {year}"
    )


    st.dataframe(
        category_spike,
        use_container_width=True,
    )


    # -----------------------------
    # Top 10 chart
    # -----------------------------

    top10 = (
        category_spike
        .head(10)
        .sort_values(
            "gmv_increase",
            ascending=True,
        )
    )


    fig, ax = plt.subplots(
        figsize=(10, 6)
    )


    bars = ax.barh(
        top10["product_category"],
        top10["gmv_increase"],
    )


    ax.set_title(
        f"Top 10 Categories Contributing to the "
        f"{comparison_name} {year} GMV Increase"
    )


    ax.set_xlabel(
        f"Incremental GMV vs {base_name} {year}"
    )


    ax.set_ylabel(
        "Product Category"
    )


    ax.grid(
        axis="x",
        alpha=0.2,
    )


    # Add contribution percentages
    for bar, pct in zip(
        bars,
        top10["contribution_to_spike_pct"],
    ):

        ax.annotate(
            f"{pct:.2f}%",
            xy=(
                bar.get_width(),
                bar.get_y()
                + bar.get_height() / 2,
            ),
            xytext=(3, 0),
            textcoords="offset points",
            va="center",
        )


    plt.tight_layout()


    st.pyplot(fig)

2026-09-15 11:54:47.582 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 11:54:47.585 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 11:54:47.586 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 11:54:47.587 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 11:54:47.587 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 11:54:47.588 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 11:54:47.590 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-15 11:54:47.592 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar